<a href="https://colab.research.google.com/github/Shayan-code04/LLM-Hallucination-Detector/blob/main/Hallucination_Detector_Colabv3_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Hallucination Detector: Evaluation on Colab (v2.2)

This notebook contains the full code, so **you don't need to upload anything**.



## 1. Install packages

In [ ]:
!pip -q install sentencepiece sentence-transformers scikit-learn requests

## 2. Write the project files

In [ ]:
%%writefile detector.py
"""
detector.py - Core hallucination-detection logic (no UI code in here).

Pipeline
--------
1. Claim extraction   : split answer into sentences, keep factual ones,
                        resolve leading pronouns ("He ..." -> "Albert Einstein ...").
2. Evidence retrieval : search Wikipedia, split pages into passages,
                        rank passages by embedding similarity to the claim.
3. Verification       : a Natural Language Inference (NLI) model decides whether the
                        evidence ENTAILS, CONTRADICTS, or is NEUTRAL to the claim.
4. Risk aggregation   : unsupported claims raise risk, uncertain claims raise it partly.

Three verification modes are kept so you can run an ablation study for the paper:
    cosine_summary : original baseline  (cosine similarity vs Wikipedia summary)
    nli_summary    : NLI vs Wikipedia summary
    nli_passage    : NLI vs best-matching passages (proposed method)
"""

from __future__ import annotations

import os
import re
import time
from dataclasses import dataclass, field, asdict
from itertools import zip_longest
from urllib.parse import quote
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import requests

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
NLI_MODELS = {
    "Small (recommended)": "cross-encoder/nli-deberta-v3-small",
    "Base (more accurate, slower)": "cross-encoder/nli-deberta-v3-base",
    "Lightweight fallback": "cross-encoder/nli-MiniLM2-L6-H768",
}
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

WIKI_API = "https://en.wikipedia.org/w/api.php"
# Wikimedia requires a descriptive User-Agent. Optionally set WIKI_CONTACT to your email.
WIKI_HEADERS = {
    "User-Agent": "HallucinationDetectorStudentProject/2.0 "
                  f"(academic prototype; contact: {os.environ.get('WIKI_CONTACT', 'student-project')})"
}

MODES = {
    "nli_passage": "NLI + passage retrieval (proposed)",
    "nli_summary": "NLI + Wikipedia summary",
    "cosine_summary": "Cosine similarity + summary (original baseline)",
}

# Starting-point thresholds. Tune them with evaluate.py --sweep.
DEFAULTS = {"t_ent": 0.70, "t_con": 0.70, "cos_hi": 0.58, "cos_lo": 0.38}

# ------------------------------------------------------------------
# Text helpers
# ------------------------------------------------------------------
SENT_SPLIT = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"\u201c])')
ENTITY_RE = re.compile(
    r"[A-Z][^\W\d_]*(?:[-'\u2019][^\W\d_]+)*"
    r"(?:\s+(?:(?:van|von|de|da|der|di|del|bin|al|of)\s+)*[A-Z][^\W\d_]*(?:[-'\u2019][^\W\d_]+)*)*"
)
ENTITY_STOP = {
    "The", "A", "An", "In", "On", "At", "It", "He", "She", "They", "This", "That",
    "These", "Those", "His", "Her", "Its", "Their", "However", "Also", "Later",
    "Then", "Today", "Moreover", "Furthermore", "Additionally", "Meanwhile",
    "Finally", "Eventually", "But", "And", "So", "As", "For", "Of", "By", "With",
}
PRON_SUBJ = re.compile(r"^(He|She|It|They)\b")
PRON_POSS = re.compile(r"^(His|Her|Its|Their)\b")
HEDGE_RE = re.compile(
    r"\b(i think|i believe|in my opinion|perhaps|maybe|might be|could be|probably)\b", re.I
)
FACT_RE = re.compile(
    r"\b(is|was|are|were|has|have|had|founded|created|located|invented|discovered|"
    r"developed|published|designed|wrote|written|born|died|built|won|named|called|known|"
    r"describes|states|explains|contains|consists|includes|means|refers|causes|produces|"
    r"orbits|flows|measures|occurs|uses|requires|allows|became|began|ended|led|used)\b",
    re.I,
)


def extract_entities(text: str) -> List[str]:
    """Very simple named-entity finder: runs of capitalised words."""
    entities = []
    for m in ENTITY_RE.finditer(text):
        tokens = m.group(0).split()
        while tokens and tokens[0] in ENTITY_STOP:
            tokens.pop(0)
        if tokens:
            entities.append(" ".join(tokens))
    return entities


def decide_nli(ent: float, con: float, t_ent: float, t_con: float) -> Tuple[str, float]:
    """Turn NLI probabilities into a verdict."""
    if ent >= t_ent and ent >= con:
        return "supported", ent
    if con >= t_con:
        return "unsupported", con
    return "uncertain", max(0.0, 1.0 - max(ent, con))


def decide_cosine(sim: float, hi: float, lo: float) -> Tuple[str, float]:
    """Original baseline decision rule (kept for the ablation)."""
    if sim >= hi:
        return "supported", min(0.95, sim + 0.15)
    if sim >= lo:
        return "uncertain", sim + 0.1
    return "unsupported", 1.0 - sim


# ------------------------------------------------------------------
# Result container
# ------------------------------------------------------------------
@dataclass
class ClaimResult:
    claim: str
    resolved_claim: str
    status: str                       # supported / unsupported / uncertain
    confidence: float
    mode: str
    entailment: Optional[float] = None
    contradiction: Optional[float] = None
    similarity: Optional[float] = None
    no_evidence: bool = False
    evidence: List[Dict] = field(default_factory=list)

    def to_dict(self) -> Dict:
        return asdict(self)


# ------------------------------------------------------------------
# Detector
# ------------------------------------------------------------------
class HallucinationDetector:
    def __init__(self, nli_model: str = NLI_MODELS["Small (recommended)"]):
        self.nli_name = nli_model
        self._embedder = None
        self._tok = None
        self._nli_model = None
        self._label_idx: Dict[str, int] = {}
        self.last_error: Optional[str] = None
        self.error_count = 0
        self._search_cache: Dict[str, List[str]] = {}
        self._page_cache: Dict[str, Optional[Dict]] = {}
        self._passage_cache: Dict[str, Tuple[List[str], np.ndarray]] = {}

    # ---------------- lazy loaders ----------------
    def _load_embedder(self):
        if self._embedder is None:
            from sentence_transformers import SentenceTransformer
            self._embedder = SentenceTransformer(EMBED_MODEL)

    def _load_nli(self):
        if self._nli_model is None:
            from transformers import AutoModelForSequenceClassification, AutoTokenizer
            self._tok = AutoTokenizer.from_pretrained(self.nli_name)
            self._nli_model = AutoModelForSequenceClassification.from_pretrained(self.nli_name).eval()
            id2label = {int(k): str(v).lower() for k, v in self._nli_model.config.id2label.items()}
            idx = {v: k for k, v in id2label.items()}
            if not {"entailment", "contradiction", "neutral"} <= set(idx):
                # cross-encoder/nli-* models use this order
                idx = {"contradiction": 0, "entailment": 1, "neutral": 2}
            self._label_idx = idx

    # ---------------- models ----------------
    def _embed(self, texts: List[str]) -> np.ndarray:
        self._load_embedder()
        return self._embedder.encode(
            texts, batch_size=64, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=False,
        )

    def nli(self, premises: List[str], hypothesis: str, batch_size: int = 8) -> List[Dict[str, float]]:
        """Return entailment / contradiction / neutral probabilities for each premise."""
        import torch
        self._load_nli()
        out = []
        for i in range(0, len(premises), batch_size):
            chunk = premises[i:i + batch_size]
            enc = self._tok(
                chunk, [hypothesis] * len(chunk), padding=True,
                truncation="only_first", max_length=512, return_tensors="pt",
            )
            with torch.no_grad():
                logits = self._nli_model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            for p in probs:
                out.append({k: float(p[self._label_idx[k]])
                            for k in ("entailment", "contradiction", "neutral")})
        return out

    # ---------------- 1. claim extraction ----------------
    def extract_claims(
        self, answer: str, question: str = "", resolve_pronouns: bool = True, max_claims: int = 12
    ) -> Tuple[List[Tuple[str, str]], bool]:
        """Return ([(original, resolved), ...], truncated_flag)."""
        text = re.sub(r"\s+", " ", answer).strip()
        sentences = [s.strip() for s in SENT_SPLIT.split(text) if s.strip()]

        q_entities = extract_entities(question)
        prev_entity = q_entities[-1] if q_entities else None

        claims: List[Tuple[str, str]] = []
        seen = set()
        for sent in sentences:
            resolved = sent
            if resolve_pronouns and prev_entity:
                if PRON_SUBJ.match(sent):
                    resolved = PRON_SUBJ.sub(prev_entity, sent, count=1)
                elif PRON_POSS.match(sent):
                    resolved = PRON_POSS.sub(prev_entity + "'s", sent, count=1)

            # remember the last entity of this sentence for the NEXT one
            ents = extract_entities(resolved)
            prev_entity = ents[-1] if ents else None

            if len(sent) < 20 or sent.endswith("?") or HEDGE_RE.search(sent):
                continue
            if any(c.isdigit() for c in sent) or FACT_RE.search(sent) or ents:
                key = resolved.lower()
                if key not in seen:
                    seen.add(key)
                    claims.append((sent, resolved))

        truncated = len(claims) > max_claims
        return claims[:max_claims], truncated

    # ---------------- 2. evidence retrieval ----------------
    def _api(self, params: Dict) -> Optional[Dict]:
        """Call the Wikipedia API directly. Records the error instead of hiding it."""
        params = {**params, "format": "json", "formatversion": "2"}
        for attempt in range(3):
            try:
                r = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=20)
                if r.status_code == 200:
                    time.sleep(0.1)   # be polite, avoids 429 rate limits
                    return r.json()
                self.last_error = f"HTTP {r.status_code}"
                if r.status_code in (429, 500, 502, 503, 504):
                    time.sleep(1.5 * (attempt + 1))
                    continue
                break
            except Exception as e:  # network error, bad JSON, timeout
                self.last_error = f"{type(e).__name__}: {e}"
                time.sleep(1.0)
        self.error_count += 1
        return None

    def check_wikipedia(self) -> Tuple[bool, Optional[str]]:
        """Quick connectivity test. Returns (ok, error_message)."""
        data = self._api({"action": "query", "list": "search", "srsearch": "Canberra", "srlimit": 1})
        ok = bool(data and data.get("query", {}).get("search"))
        return ok, (None if ok else (self.last_error or "empty response from Wikipedia"))

    def _search(self, query: str) -> List[str]:
        if query not in self._search_cache:
            data = self._api({"action": "query", "list": "search",
                              "srsearch": query, "srlimit": 3})
            if data is None:            # do not cache failures
                return []
            self._search_cache[query] = [h["title"] for h in data.get("query", {}).get("search", [])]
        return self._search_cache[query]

    def _page(self, title: str) -> Optional[Dict]:
        if title not in self._page_cache:
            data = self._api({"action": "query", "prop": "extracts", "explaintext": 1,
                              "redirects": 1, "titles": title})
            if data is None:            # do not cache failures
                return None
            pages = data.get("query", {}).get("pages", [])
            p = pages[0] if pages else {}
            text = p.get("extract") or ""
            if p.get("missing") or not text.strip() or "may refer to" in text[:300].lower():
                self._page_cache[title] = None
            else:
                self._page_cache[title] = {
                    "title": p["title"],
                    "url": "https://en.wikipedia.org/wiki/" + quote(p["title"].replace(" ", "_")),
                    "summary": re.split(r"\n\s*==", text, maxsplit=1)[0].strip(),
                    "content": text[:30000],
                }
        return self._page_cache[title]

    def retrieve_pages(self, claim: str, max_pages: int = 5) -> List[Dict]:
        """Exact-title pages for entities in the claim first, then search results."""
        ents = list(dict.fromkeys(extract_entities(claim)))
        pages: List[Dict] = []
        seen = set()

        def add(page):
            if page and page["title"] not in seen:
                seen.add(page["title"])
                pages.append(page)

        for e in ents[:3]:                      # 1) direct lookup, e.g. "Eiffel Tower"
            add(self._page(e))

        clean = re.sub(r"[^\w\s]", " ", claim)
        clean = re.sub(r"\s+", " ", clean).strip()[:150]
        queries = [clean] + ([" ".join(ents)] if ents else [])
        titles: List[str] = []
        for group in zip_longest(*[self._search(q) for q in queries]):
            for t in group:
                if t and t not in titles:
                    titles.append(t)
        for t in titles:                        # 2) search results fill the rest
            if len(pages) >= max_pages:
                break
            add(self._page(t))
        return pages

    def _page_passages(self, page: Dict) -> Tuple[List[str], np.ndarray]:
        title = page["title"]
        if title not in self._passage_cache:
            lines = [l.strip() for l in page["content"].split("\n")
                     if l.strip() and not l.strip().startswith("=")]
            sents = [s for s in SENT_SPLIT.split(" ".join(lines)) if len(s) > 20]
            passages = [" ".join(sents[i:i + 2]) for i in range(max(1, len(sents) - 1))]
            passages = [p for p in passages if p][:400]
            emb = self._embed(passages) if passages else np.zeros((0, 384), dtype="float32")
            self._passage_cache[title] = (passages, emb)
        return self._passage_cache[title]

    # ---------------- 3. verification ----------------
    def verify_claim(
        self, claim: str, resolved: Optional[str] = None, mode: str = "nli_passage",
        top_k: int = 3, t_ent: float = DEFAULTS["t_ent"], t_con: float = DEFAULTS["t_con"],
        cos_hi: float = DEFAULTS["cos_hi"], cos_lo: float = DEFAULTS["cos_lo"],
        pages: Optional[List[Dict]] = None,
    ) -> ClaimResult:
        text = resolved or claim
        if pages is None:
            pages = self.retrieve_pages(text)
        if not pages:
            return ClaimResult(claim, text, "uncertain", 0.0, mode, no_evidence=True)

        # ----- original baseline: cosine vs 600-char summary -----
        if mode == "cosine_summary":
            claim_emb = self._embed([text])[0]
            summaries = [p["summary"][:600] for p in pages]
            sims = self._embed(summaries) @ claim_emb
            best = int(np.argmax(sims))
            status, conf = decide_cosine(float(sims[best]), cos_hi, cos_lo)
            ev = [{"text": summaries[best][:400], "source": pages[best]["title"],
                   "url": pages[best]["url"], "similarity": float(sims[best])}]
            return ClaimResult(claim, text, status, round(conf, 3), mode,
                               similarity=float(sims[best]), evidence=ev)

        # ----- choose premises -----
        if mode == "nli_summary":
            cands = [{"text": p["summary"][:600], "source": p["title"],
                      "url": p["url"], "similarity": None} for p in pages]
        else:  # nli_passage
            claim_emb = self._embed([text])[0]
            pool = []
            for p in pages:
                passages, emb = self._page_passages(p)
                if len(passages) == 0:
                    continue
                sims = emb @ claim_emb
                for j in np.argsort(-sims)[:top_k]:
                    pool.append({"text": passages[j], "source": p["title"],
                                 "url": p["url"], "similarity": float(sims[j])})
            pool.sort(key=lambda d: -d["similarity"])
            cands = pool[:top_k]

        if not cands:
            return ClaimResult(claim, text, "uncertain", 0.0, mode, no_evidence=True)

        scores = self.nli([c["text"] for c in cands], text)
        for c, s in zip(cands, scores):
            c["entailment"], c["contradiction"] = s["entailment"], s["contradiction"]

        ent = max(c["entailment"] for c in cands)
        con = max(c["contradiction"] for c in cands)
        status, conf = decide_nli(ent, con, t_ent, t_con)
        cands.sort(key=lambda c: -(c["entailment"] if status == "supported" else c["contradiction"]))
        return ClaimResult(claim, text, status, round(conf, 3), mode,
                           entailment=ent, contradiction=con, evidence=cands)

    # ---------------- 4. full pipeline ----------------
    def analyze(
        self, question: str, answer: str, mode: str = "nli_passage", top_k: int = 3,
        t_ent: float = DEFAULTS["t_ent"], t_con: float = DEFAULTS["t_con"],
        resolve_pronouns: bool = True, max_claims: int = 12,
        progress: Optional[Callable[[int, int], None]] = None,
    ) -> Dict:
        claims, truncated = self.extract_claims(answer, question, resolve_pronouns, max_claims)
        if not claims:
            return {
                "risk_score": None, "risk_level": "Not enough claims", "claims": [],
                "supported": 0, "unsupported": 0, "uncertain": 0, "truncated": False,
                "summary": "No checkable factual claims were found, so no risk score is given. "
                           "This does NOT mean the answer is correct.",
            }

        results = []
        for i, (orig, resolved) in enumerate(claims, 1):
            results.append(self.verify_claim(orig, resolved, mode, top_k, t_ent, t_con))
            if progress:
                progress(i, len(claims))

        warning = None
        if all(r.no_evidence for r in results):
            warning = ("No Wikipedia evidence was retrieved for ANY claim, so these results are not "
                       f"meaningful. Possible network problem. Last error: {self.last_error}")

        s = sum(r.status == "supported" for r in results)
        u = sum(r.status == "unsupported" for r in results)
        c = sum(r.status == "uncertain" for r in results)
        total = len(results)
        risk = round((u * 1.0 + c * 0.4) / total, 3)
        level = "Low" if risk < 0.30 else "Medium" if risk < 0.60 else "High"

        summary = (f"Analyzed {total} claims: {s} supported, {u} unsupported (contradicted by "
                   f"evidence), {c} uncertain. Overall hallucination risk is **{level}**.")
        if truncated:
            summary += f" Only the first {max_claims} claims were checked."
        return {
            "risk_score": risk, "risk_level": level, "claims": results, "summary": summary,
            "supported": s, "unsupported": u, "uncertain": c, "truncated": truncated,
            "warning": warning,
        }


Writing detector.py


In [ ]:
%%writefile evaluate.py
"""
evaluate.py - Measure how well each verification method detects false claims.

Usage:
    python evaluate.py                       # default thresholds, small NLI model
    python evaluate.py --sweep               # also search for the best thresholds
    python evaluate.py --nli base --sweep    # use the bigger NLI model
    python evaluate.py --data my_claims.csv  # your own file (columns: claim,label ; label = true/false)

Positive class = "hallucination" (label false).
Two operating points are reported:
    strict : only claims marked 'unsupported' are flagged
    flag   : 'unsupported' AND 'uncertain' claims are flagged (safer, more false alarms)

Outputs: results_summary.csv (paper table) and results_per_claim.csv (error analysis).
"""

import argparse
import csv
import itertools
import sys
import time

import numpy as np
from sklearn.metrics import roc_auc_score

from detector import (DEFAULTS, MODES, NLI_MODELS, HallucinationDetector,
                      decide_cosine, decide_nli)

NLI_CHOICES = {"small": "Small (recommended)", "base": "Base (more accurate, slower)",
               "light": "Lightweight fallback"}


def load_data(path):
    claims, y = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            lab = row["label"].strip().lower()
            if lab not in ("true", "false"):
                raise ValueError(f"label must be true/false, got: {row['label']}")
            claims.append(row["claim"].strip())
            y.append(1 if lab == "false" else 0)       # 1 = hallucination
    return claims, np.array(y)


def status_of(mode, raw, thr):
    if raw["no_evidence"]:
        return "uncertain"
    if mode == "cosine_summary":
        return decide_cosine(raw["sim"], thr["cos_hi"], thr["cos_lo"])[0]
    return decide_nli(raw["ent"], raw["con"], thr["t_ent"], thr["t_con"])[0]


def halluc_score(mode, raw):
    """Continuous score (higher = more likely hallucination) for AUROC."""
    if raw["no_evidence"]:
        return 0.0
    if mode == "cosine_summary":
        return -raw["sim"]
    return raw["con"] - raw["ent"]


def prf(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def evaluate_mode(mode, raws, y, thr):
    st = np.array([status_of(mode, r, thr) for r in raws])
    rows = {}
    for op, flagged in (("strict", ["unsupported"]), ("flag", ["unsupported", "uncertain"])):
        pred = np.isin(st, flagged).astype(int)
        p, r, f = prf(y, pred)
        rows[op] = (p, r, f)
    decided = st != "uncertain"
    acc = (((st == "unsupported").astype(int) == y)[decided].mean()) if decided.any() else 0.0
    cov = decided.mean()
    scores = np.array([halluc_score(mode, r) for r in raws])
    auroc = roc_auc_score(y, scores) if len(set(y)) > 1 else float("nan")
    return rows, acc, cov, auroc, st


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="test_set.csv")
    ap.add_argument("--nli", choices=list(NLI_CHOICES), default="small")
    ap.add_argument("--sweep", action="store_true", help="search thresholds for best F1 (flag mode)")
    ap.add_argument("--split", action="store_true",
                    help="with --sweep: tune thresholds on one half, report on the held-out half")
    ap.add_argument("--top_k", type=int, default=3)
    args = ap.parse_args()

    claims, y = load_data(args.data)
    print(f"Loaded {len(claims)} claims  ({int(y.sum())} false / {int((1 - y).sum())} true)")

    det = HallucinationDetector(NLI_MODELS[NLI_CHOICES[args.nli]])
    ok, msg = det.check_wikipedia()
    if not ok:
        print("\nERROR: cannot reach Wikipedia (" + str(msg) + "). Results would be meaningless, stopping.")
        sys.exit(1)
    print("Wikipedia connection OK.\n")

    raw = {m: [] for m in MODES}
    for i, claim in enumerate(claims, 1):
        print(f"[{i}/{len(claims)}] {claim[:70]}")
        pages = det.retrieve_pages(claim)          # retrieve ONCE so every method sees the same evidence
        for attempt in range(3):
            if pages:
                break
            time.sleep(2.0)
            pages = det.retrieve_pages(claim)
        for m in MODES:
            r = det.verify_claim(claim, None, m, top_k=args.top_k, pages=pages)
            raw[m].append({"ent": r.entailment or 0.0, "con": r.contradiction or 0.0,
                           "sim": r.similarity or 0.0, "no_evidence": r.no_evidence,
                           "source": r.evidence[0]["source"] if r.evidence else ""})

    no_ev = np.mean([r["no_evidence"] for r in raw["nli_passage"]])
    if no_ev > 0.3:
        print(f"\nWARNING: no evidence was found for {no_ev:.0%} of claims. Check results carefully.")

    summary, per_claim = [], {m: {} for m in MODES}
    print("\n=== Results (default thresholds) ===")
    print(f"{'method':<50}{'op':<8}{'P':>6}{'R':>6}{'F1':>6}{'acc*':>7}{'cov':>6}{'AUROC':>7}")
    for m in MODES:
        rows, acc, cov, auroc, st = evaluate_mode(m, raw[m], y, DEFAULTS)
        per_claim[m]["default"] = st
        for op, (p, r, f) in rows.items():
            print(f"{MODES[m]:<50}{op:<8}{p:>6.2f}{r:>6.2f}{f:>6.2f}{acc:>7.2f}{cov:>6.2f}{auroc:>7.2f}")
            summary.append([MODES[m], "default", op, f"{p:.3f}", f"{r:.3f}", f"{f:.3f}",
                            f"{acc:.3f}", f"{cov:.3f}", f"{auroc:.3f}",
                            str({k: DEFAULTS[k] for k in DEFAULTS})])

    if args.sweep:
        if args.split:
            rng = np.random.default_rng(0)
            tune, test = [], []
            for lab in (0, 1):                       # stratified split
                idx = np.where(y == lab)[0]
                rng.shuffle(idx)
                h = len(idx) // 2
                tune += list(idx[:h])
                test += list(idx[h:])
            tune, test = np.array(sorted(tune)), np.array(sorted(test))
            tag = "tuned_on_half_reported_on_heldout"
            print(f"\n=== Thresholds tuned on {len(tune)} claims, reported on {len(test)} held-out claims ===")
        else:
            tune = test = np.arange(len(y))
            tag = "tuned_same_data_optimistic"
            print("\n=== Best thresholds by F1 (flag mode) - tuned on the same data, so OPTIMISTIC ===")

        for m in MODES:
            if m == "cosine_summary":
                grid = [dict(DEFAULTS, cos_hi=round(float(h), 2), cos_lo=round(float(h) - d, 2))
                        for h in np.arange(0.3, 0.85, 0.05) for d in (0.1, 0.2, 0.3)]
                keys = ("cos_hi", "cos_lo")
            else:
                grid = [dict(DEFAULTS, t_ent=a, t_con=b)
                        for a, b in itertools.product([0.5, 0.6, 0.7, 0.8, 0.9], repeat=2)]
                keys = ("t_ent", "t_con")
            best = None
            for thr in grid:
                rows, *_ = evaluate_mode(m, [raw[m][i] for i in tune], y[tune], thr)
                if best is None or rows["flag"][2] > best[0]:
                    best = (rows["flag"][2], thr)
            thr = best[1]
            rows, acc, cov, auroc, _ = evaluate_mode(m, [raw[m][i] for i in test], y[test], thr)
            per_claim[m]["tuned"] = evaluate_mode(m, raw[m], y, thr)[4]
            print(f"{MODES[m]:<50} thresholds {[thr[k] for k in keys]}  "
                  f"F1(flag)={rows['flag'][2]:.2f}  F1(strict)={rows['strict'][2]:.2f}  AUROC={auroc:.2f}")
            for op, (p, r, f) in rows.items():
                summary.append([MODES[m], tag, op, f"{p:.3f}", f"{r:.3f}", f"{f:.3f}",
                                f"{acc:.3f}", f"{cov:.3f}", f"{auroc:.3f}",
                                str({k: thr[k] for k in keys})])

    print("\n* acc = accuracy on claims that got a decision (not 'uncertain'); cov = share of claims decided.")

    with open("results_summary.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["method", "thresholds_set", "operating_point", "precision", "recall", "f1",
                    "accuracy_on_decided", "coverage", "auroc", "thresholds"])
        w.writerows(summary)

    with open("results_per_claim.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        header = ["claim", "gold"]
        for m in MODES:
            header += [f"{m}_status", f"{m}_top_source"]
        w.writerow(header)
        for i, c in enumerate(claims):
            row = [c, "false" if y[i] else "true"]
            for m in MODES:
                row += [per_claim[m]["default"][i], raw[m][i]["source"]]
            w.writerow(row)
    print("Saved results_summary.csv and results_per_claim.csv")


if __name__ == "__main__":
    main()


Writing evaluate.py


In [ ]:
%%writefile test_set.csv
claim,label
Canberra is the capital city of Australia.,true
Albert Einstein published the theory of general relativity in 1915.,true
Alexander Graham Bell was awarded the first US patent for the telephone in 1876.,true
The Eiffel Tower is located in Paris France.,true
Mount Everest is the highest mountain above sea level.,true
William Shakespeare wrote the play Hamlet.,true
Tim Berners-Lee invented the World Wide Web while working at CERN.,true
The Pacific Ocean is the largest ocean on Earth.,true
Marie Curie won Nobel Prizes in both Physics and Chemistry.,true
The Sydney Opera House was designed by the Danish architect Jørn Utzon.,true
Tokyo is the capital of Japan.,true
Isaac Newton formulated the laws of motion and universal gravitation.,true
The Amazon River flows through South America.,true
Leonardo da Vinci painted the Mona Lisa.,true
The Berlin Wall fell in 1989.,true
Python was created by Guido van Rossum.,true
Neil Armstrong was the first person to walk on the Moon in 1969.,true
The chemical symbol for gold is Au.,true
Brasília is the capital of Brazil.,true
The Taj Mahal is located in Agra India.,true
Sydney is the capital city of Australia.,false
Nikola Tesla invented the telephone in 1893.,false
The Eiffel Tower is located in Rome Italy.,false
Mount Kilimanjaro is the highest mountain in the world.,false
Charles Dickens wrote the play Hamlet.,false
Thomas Edison invented the World Wide Web at CERN.,false
The Atlantic Ocean is the largest ocean on Earth.,false
Marie Curie won a Nobel Prize in Literature.,false
The Sydney Opera House was designed by Frank Lloyd Wright.,false
Kyoto is the current capital of Japan.,false
Galileo Galilei formulated the theory of general relativity.,false
The Amazon River flows through Africa.,false
Vincent van Gogh painted the Mona Lisa.,false
The Berlin Wall fell in 1975.,false
Python was created by James Gosling.,false
Yuri Gagarin was the first person to walk on the Moon in 1969.,false
The chemical symbol for gold is Ag.,false
Rio de Janeiro is the capital of Brazil.,false
The Taj Mahal is located in Delhi India.,false
Albert Einstein published the theory of general relativity in 1895.,false


Writing test_set.csv


## 3. (Optional) Use your own, bigger test set
CSV with columns `claim` and `label` (`true` or `false`). **Skip this cell to use the built-in 40 claims**; if you run it, choose a file or click Cancel.

In [ ]:
from google.colab import files
import shutil
up = files.upload()
for name in up:
    shutil.move(name, "test_set.csv")
    print("Using", name, "as test_set.csv")

## 4. Check that Wikipedia can be reached


In [ ]:
from detector import HallucinationDetector
t = HallucinationDetector()
ok, msg = t.check_wikipedia()
print("Wikipedia connection:", "OK" if ok else f"FAILED -> {msg}")
if ok:
    pages = t.retrieve_pages("Sydney is the capital city of Australia.")
    print("Pages retrieved for a test claim:", [p["title"] for p in pages])
    assert pages, "Search worked but no pages could be loaded"

ModuleNotFoundError: No module named 'detector'

## 5. Run the evaluation
Compares: cosine baseline, NLI on summaries, NLI on retrieved passages (proposed). All three methods now see **identical Wikipedia evidence**. `--split` tunes thresholds on half the claims and reports on the held-out half. Takes several minutes.

In [ ]:
!python evaluate.py --nli small --sweep --split

Loaded 40 claims  (20 false / 20 true)
Wikipedia connection OK.

[1/40] Canberra is the capital city of Australia.
modules.json: 100% 349/349 [00:00<00:00, 1.24MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 369kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 23.3MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 189kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.65MB/s]

model.safetensors: downloading bytes:  88% 80.2M/90.9M [00:00<00:00, 128MB/s, 6.28MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 74.1MB/s, 8.08MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 79.2MB/s, 8.82MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 3375.66it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.35MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 14.4MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 41.0MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 394kB/s]
config.json: 100% 190/190 [00:00<00:00

## 6. Print the results so you can copy them (no download needed)

In [ ]:
print(open("results_summary.csv", encoding="utf-8").read())
print("=" * 60)
print(open("results_per_claim.csv", encoding="utf-8").read())

method,thresholds_set,operating_point,precision,recall,f1,accuracy_on_decided,coverage,auroc,thresholds
NLI + passage retrieval (proposed),default,strict,0.882,0.750,0.811,0.861,0.900,0.905,"{'t_ent': 0.7, 't_con': 0.7, 'cos_hi': 0.58, 'cos_lo': 0.38}"
NLI + passage retrieval (proposed),default,flag,0.810,0.850,0.829,0.861,0.900,0.905,"{'t_ent': 0.7, 't_con': 0.7, 'cos_hi': 0.58, 'cos_lo': 0.38}"
NLI + Wikipedia summary,default,strict,0.783,0.900,0.837,0.833,0.900,0.902,"{'t_ent': 0.7, 't_con': 0.7, 'cos_hi': 0.58, 'cos_lo': 0.38}"
NLI + Wikipedia summary,default,flag,0.704,0.950,0.809,0.833,0.900,0.902,"{'t_ent': 0.7, 't_con': 0.7, 'cos_hi': 0.58, 'cos_lo': 0.38}"
Cosine similarity + summary (original baseline),default,strict,0.000,0.000,0.000,0.500,1.000,0.863,"{'t_ent': 0.7, 't_con': 0.7, 'cos_hi': 0.58, 'cos_lo': 0.38}"
Cosine similarity + summary (original baseline),default,flag,0.000,0.000,0.000,0.500,1.000,0.863,"{'t_ent': 0.7, 't_con': 0.7, 'cos_hi': 0.58, 'cos_lo': 0.38}"
NLI 

## 7. Try your own LLM answer

In [ ]:
det = HallucinationDetector()
question = "What is the capital of india and when was it founded?"
answer = ("The capital of india is delhi. It was founded in 1788 by Captain James Cook. "
          "Sydney is located on the southeastern coast and is the largest city in the country. "
          "The famous Sydney Opera House was designed by Jørn Utzon.")
res = det.analyze(question, answer)
if res.get("warning"): print("WARNING:", res["warning"])
print(res["summary"], "\n")
for c in res["claims"]:
    ent = "n/a" if c.entailment is None else f"{c.entailment:.2f}"
    con = "n/a" if c.contradiction is None else f"{c.contradiction:.2f}"
    src = c.evidence[0]["source"] if c.evidence else "none"
    print(f"[{c.status.upper():11}] {c.resolved_claim}")
    print(f"              entail={ent}  contradict={con}  top source: {src}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Analyzed 4 claims: 3 supported, 1 unsupported (contradicted by evidence), 0 uncertain. Overall hallucination risk is **Low**. 

[SUPPORTED  ] The capital of india is delhi.
              entail=1.00  contradict=0.02  top source: Delhi

[UNSUPPORTED] It was founded in 1788 by Captain James Cook.
              entail=0.00  contradict=1.00  top source: First voyage of James Cook

[SUPPORTED  ] Sydney is located on the southeastern coast and is the largest city in the country.
              entail=0.93  contradict=0.00  top source: Sydney

[SUPPORTED  ] The famous Sydney Opera House was designed by Jørn Utzon.
              entail=0.99  contradict=0.07  top source: Wave House

